### Import Library

In [1]:
import polars as pl
from pathlib import Path
import fastexcel
import pandas as pd
import xlsxwriter

### Function Import Excel

In [2]:
def read_excel(path: Path, sheet_pattern: str | None = None, had_header: bool = True) -> pl.DataFrame:
    """
    Read matching sheets from an Excel file into a single Polars DataFrame.
    
    - sheet_pattern: prefix to match sheet names (e.g. "Income" matches "Income - 1", "Income - 2").
                     None = read all sheets.
    - had_header: whether the sheet has a header row.
    """
    reader = fastexcel.read_excel(path)
    print(f"  Sheets in {path.name}: {reader.sheet_names}")

    # Filter sheets by pattern
    sheets = [s for s in reader.sheet_names
              if sheet_pattern is None or s.startswith(sheet_pattern)]
    print(f"  Matching sheets: {sheets}")

    dfs = []
    for name in sheets:
        df = pl.read_excel(path, sheet_name=name, has_header=had_header)
        df = df.with_columns(pl.lit(name).alias("_source_sheet"))
        dfs.append(df)

    if not dfs:
        return pl.DataFrame()

    combined = pl.concat(dfs, how="diagonal_relaxed") if len(dfs) > 1 else dfs[0]
    return combined.with_columns(pl.lit(path.name).alias("_source_file"))

In [3]:
def concat_files(files: list[Path], label: str, sheet_pattern: str | int | None = None, has_header: bool = True) -> pl.DataFrame:
    if not files:
        return pl.DataFrame()
    dfs = []
    for f in files:
        df = read_excel(f, sheet_pattern=sheet_pattern, had_header=has_header)
        if label == "income_all" and not has_header:
            header = df.row(2)
            df = df.slice(3)
            df.columns = header
            dfs.append(df)
        elif label == "balance_all" and not has_header:
            header =df.row(13)
            df = df.slice(14)
            df.columns = header
            dfs.append(df)
        print(f"  Loaded {f.name}: {df.shape}")
    combined = pl.concat(dfs, how="diagonal_relaxed")
    # before = combined.height
    # Drop duplicates (exclude _source_file so rows from overlapping files are caught)
    # data_cols = [c for c in combined.columns if c != "_source_file"]
    # combined = combined.unique(subset=data_cols, keep="first")
    # after = combined.height
    # dupes = before - after
    # print(f"=> {label}: {combined.shape} (removed {dupes} duplicate rows)\n")
    return combined

### Assign File Location 

In [4]:
root = Path(r"c:\Users\TanJunJie\OneDrive - SRKK Group\Project\watson_entriesmatching\OneDrive_2026-03-09\Shopee Sample Reports (Testing)\scenario2")

# Find all .xlsx files recursively
xlsx_files = sorted(root.rglob("*.xlsx"))

### Income Report


In [5]:
# --- Load & concatenate by report type ---
income_files = [f for f in xlsx_files if f.name.startswith("Income.released")]
print("=== Income Released ===")
income_all = concat_files(income_files, "income_all", sheet_pattern="Income", has_header=False)



=== Income Released ===
  Sheets in Income.released.my.20251231_20251231.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Income - 1', 'Adjustment', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20251231_20251231.xlsx: (15021, 49)
  Sheets in Income.released.my.20260101_20260101.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260101_20260101.xlsx: (12599, 49)
  Sheets in Income.released.my.20260102_20260102.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260102_20260102.xlsx: (16012, 49)
  Sheets in Income.released.my.20260103_20260103.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']


In [6]:
dup_order_id = (
income_all
.group_by("Order ID")
.len()
.filter(pl.col("len") > 1)
.height
)
print("duplicate Order ID groups:", dup_order_id)

duplicate Order ID groups: 71514


In [7]:
#Deduplication of entries, remove duplicates based on "Order ID" column, keep the first occurrence and maintain the original order
income_all = income_all.unique(
    subset=["Order ID"],
    keep="first",
    maintain_order=True
)

In [8]:
dup_order_id = (
income_all
.group_by("Order ID")
.len()
.filter(pl.col("len") > 1)
.height
)
print("duplicate Order ID groups:", dup_order_id)

duplicate Order ID groups: 0


In [9]:
 #Select only relevant columns for matching (exclude "View By" which is used for filtering but not needed in matching)
income_all_normalized = (
    income_all.
    filter(pl.col("View By")== "Order")
    .select(
    ['Sequence No.',
    'Order ID',
    'Order Creation Date',
    'Payout Completed Date',

    'Total Released Amount (RM)',
    'Product Price',
    'Refund Amount',
    'Shipping Fee Paid by Buyer (excl. SST)',
    'Shipping Fee Charged by Logistic Provider',
    'Seller Paid Shipping Fee SST',
    'Shipping Rebate From Shopee',
    'Reverse Shipping Fee',
    'Reverse Shipping Fee SST',
    'Saver Programme Shipping Fee Savings',
    'Return to Seller Fee',
    'Rebate Provided by Shopee',
    'Voucher Sponsored by Seller',
    'Coin Cashback Sponsored by Seller',
    'Commission Fee (incl. SST)',
    'Service Fee (Incl. SST)',
    'Transaction Fee (Incl. SST)',
    'AMS Commission Fee',
    'Saver Programme Fee (Incl. SST)',

    'Username (Buyer)',
    'Transaction Fee Rate (%)',
    'Buyer Payment Method',
    'Buyer Payment Method Details_1(if applicable)',
    'Payment Details / Installment Plan',

    'Voucher Code From Seller',
    'Lost Compensation',
    ])
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col('Total Released Amount (RM)').cast(pl.Float64, strict=False),
        pl.col('Order Creation Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Payout Completed Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Product Price').cast(pl.Float64, strict=False),
        pl.col('Refund Amount').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Paid by Buyer (excl. SST)').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Charged by Logistic Provider').cast(pl.Float64, strict=False),
        pl.col('Seller Paid Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Shipping Rebate From Shopee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Shipping Fee Savings').cast(pl.Float64, strict=False),
        pl.col('Return to Seller Fee').cast(pl.Float64, strict=False),
        pl.col('Rebate Provided by Shopee').cast(pl.Float64, strict=False),
        pl.col('Voucher Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Coin Cashback Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Commission Fee (incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Service Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('AMS Commission Fee').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee Rate (%)').cast(pl.Float64, strict=False),
        pl.col('Lost Compensation').cast(pl.Float64, strict=False),
    ])
    .drop_nulls(['Order ID'])
    )
income_all_normalized.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64
"""1""","""251230QHF7YGBE""",2025-12-30,2025-12-31,-5.19,27.93,-27.93,0.0,-4.9,-0.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ndkshidshisbssayacantik""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0
"""3""","""251230PMP6MTW6""",2025-12-30,2025-12-31,-2.65,14.9,-14.9,0.0,-2.5,-0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gq6pe2qreq""",3.78,"""Online Banking""","""""","""""","""""",0.0
"""5""","""251230PB5YG8HT""",2025-12-30,2025-12-31,37.32,44.1,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.64,0.0,-2.14,0.0,0.0,"""fiqamokhtar""",4.86,"""SPayLater""","""""","""12x""","""""",0.0
"""7""","""251230PB3GD63E""",2025-12-30,2025-12-31,34.51,41.04,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.54,0.0,-1.99,0.0,0.0,"""nurnabilahabukasim""",4.86,"""SPayLater""","""""","""3x""","""""",0.0
"""10""","""251230PAY9XQWY""",2025-12-30,2025-12-31,26.79,31.5,0.0,4.9,-4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.32,0.0,-1.39,0.0,0.0,"""amyrahbrown""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0


### Balance Report 

In [10]:
balance_files = [f for f in xlsx_files if f.name.startswith("my_balance_transaction")]
print("=== Balance Transaction ===")
balance_all = concat_files(balance_files, "balance_all", sheet_pattern="Transaction", has_header=False)

=== Balance Transaction ===
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: (5000, 10)
  Sheets

In [11]:
dup_order_id = (
balance_all
.group_by("Order ID")
.len()
.filter(pl.col("len") > 1)
.height
)
print("duplicate Order ID groups:", dup_order_id)

duplicate Order ID groups: 5563


In [12]:
#Deduplication of entries, remove duplicates based on "Order ID" column, keep the first occurrence and maintain the original order
balance_all = balance_all.unique(
    subset=["Order ID"],
    keep="first",
    maintain_order=True
)

In [13]:
dup_full_rows = balance_all.is_duplicated().sum()
print("exact duplicate rows:", dup_full_rows)

exact duplicate rows: 0


In [14]:
balance_all_normalized = (
    balance_all
    .select(['Date',
        'Transaction Type',
        'Description',
        'Order ID',
        'Money Direction',
        'Amount',
        'Status',
        'Balance After Transactions',
        'Transaction Report'])
    # Step 1: parse/cast raw types first
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col('Date').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S'),
        pl.col('Amount').cast(pl.Float64, strict=False).alias('Payment Amount'),
        pl.col('Balance After Transactions').cast(pl.Float64, strict=False),
    ])
    # Step 2: derive columns from the now-Datetime "Date"
    .with_columns([
        pl.col('Date').dt.strftime('%d-%b-%Y').str.to_uppercase().alias('Payment Date'),
        pl.col('Date').dt.strftime('%b-%Y').str.to_uppercase().alias('Payment Month'),
    ])
)
balance_all_normalized.head(10)

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,Payment Amount,Payment Date,Payment Month
datetime[μs],str,str,str,str,str,str,f64,str,f64,str,str
2026-01-07 23:59:53,"""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP""","""Money In""","""13.42""","""Transaction Completed""",195391.02,"""Transaction Report""",13.42,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:46,"""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV""","""Money In""","""8.31""","""Transaction Completed""",195377.6,"""Transaction Report""",8.31,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:32,"""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG""","""Money In""","""21.25""","""Transaction Completed""",195369.29,"""Transaction Report""",21.25,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:28,"""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6""","""Money In""","""13.19""","""Transaction Completed""",195348.04,"""Transaction Report""",13.19,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:26,"""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU""","""Money In""","""23.08""","""Transaction Completed""",195334.85,"""Transaction Report""",23.08,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:06,"""Order Income""","""Income from Order #26010575HF5…","""26010575HF5Q6D""","""Money In""","""14.92""","""Transaction Completed""",195311.77,"""Transaction Report""",14.92,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:02,"""Order Income""","""Income from Order #26010347C47…","""26010347C470NX""","""Money In""","""16.71""","""Transaction Completed""",195296.85,"""Transaction Report""",16.71,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:58:33,"""Order Income""","""Income from Order #260101UAHS8…","""260101UAHS8FRR""","""Money In""","""6.76""","""Transaction Completed""",195280.14,"""Transaction Report""",6.76,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:58:22,"""Order Income""","""Income from Order #2601069VSE7…","""2601069VSE7UX0""","""Money In""","""33.34""","""Transaction Completed""",195273.38,"""Transaction Report""",33.34,"""07-JAN-2026""","""JAN-2026"""


In [15]:
balance_all_normalized.head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,Payment Amount,Payment Date,Payment Month
datetime[μs],str,str,str,str,str,str,f64,str,f64,str,str
2026-01-07 23:59:53,"""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP""","""Money In""","""13.42""","""Transaction Completed""",195391.02,"""Transaction Report""",13.42,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:46,"""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV""","""Money In""","""8.31""","""Transaction Completed""",195377.6,"""Transaction Report""",8.31,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:32,"""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG""","""Money In""","""21.25""","""Transaction Completed""",195369.29,"""Transaction Report""",21.25,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:28,"""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6""","""Money In""","""13.19""","""Transaction Completed""",195348.04,"""Transaction Report""",13.19,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:26,"""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU""","""Money In""","""23.08""","""Transaction Completed""",195334.85,"""Transaction Report""",23.08,"""07-JAN-2026""","""JAN-2026"""


### Sales Report

In [16]:
sales_files = sorted(root.parent.glob("SalesReport*.xlsx"))
print("=== Sales Reports ===")
print([f.name for f in sales_files])

sales_dfs = []
for f in sales_files:
    df = pl.read_excel(f, sheet_name="SalesReport", has_header=True)
    df = df.with_columns(pl.lit(f.name).alias("_source_file"))
    sales_dfs.append(df)

sales_report = pl.concat(sales_dfs, how="diagonal_relaxed") if sales_dfs else pl.DataFrame()
print("sales_report shape:", sales_report.shape)

sales_report.head()

=== Sales Reports ===
['SalesReport Wk2.xlsx', 'SalesReport Wk3.xlsx']


Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string


sales_report shape: (62427, 10)


OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file
str,str,i64,i64,date,f64,str,str,str,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""


In [17]:
# Normalize sales report
sales_report_normalized = (
    sales_report
    .with_columns([
        # `SalesWorkDate` may already be a Date/Datetime (from Excel reader),
        # or it may be a string like "31-12-2025"; handle both safely.
        pl.coalesce([
            pl.col("SalesWorkDate").cast(pl.Date, strict=False),
            pl.col("SalesWorkDate")
              .cast(pl.Utf8, strict=False)
              .str.strip_chars()
              .str.strptime(pl.Date, "%d-%m-%Y", strict=False),
        ]).alias("WorkDate"),
        pl.col("MarketPlaceOrderNum").cast(pl.Utf8).str.strip_chars().alias("Order ID"),
        pl.col("TotalAmount").cast(pl.Float64, strict=False).alias("SalesCenterAmount"),
        
    ])
    .with_columns([
        pl.col("WorkDate").dt.strftime("%b-%Y").str.to_uppercase().alias("Sales Month")
    ])
)
sales_report_normalized.head()

OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file,WorkDate,Order ID,SalesCenterAmount,Sales Month
str,str,i64,i64,date,f64,str,str,str,str,date,str,f64,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""2601044PUM2781""",21.0,"""JAN-2026"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""26010450MFPQE1""",98.29,"""JAN-2026"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""2601045R7NTP28""",107.65,"""JAN-2026"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""260101TUY6HNTV""",9.56,"""JAN-2026"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""260101UF58WU38""",2.73,"""JAN-2026"""


# Generate Report

## Generate Payment Summary

In [18]:
# Add "Net Voucher" column as sum of all voucher-related columns (treating nulls as 0)
income_all_normalized = income_all_normalized.with_columns(
    pl.sum_horizontal([
        pl.col('Rebate Provided by Shopee'),
        pl.col('Voucher Sponsored by Seller'),
        pl.col('Coin Cashback Sponsored by Seller')
    ]).fill_null(0).alias("Net Voucher")
)

In [19]:
income_all_normalized = income_all_normalized.with_columns(
    pl.sum_horizontal([
        pl.col('Shipping Fee Paid by Buyer (excl. SST)'),
        pl.col('Shipping Fee Charged by Logistic Provider'),
        pl.col('Seller Paid Shipping Fee SST'),
        pl.col('Shipping Rebate From Shopee'),
        pl.col('Reverse Shipping Fee'),
        pl.col('Reverse Shipping Fee SST')
    ]).fill_null(0).alias("Net Shipping Fees")
)
income_all_normalized.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""1""","""251230QHF7YGBE""",2025-12-30,2025-12-31,-5.19,27.93,-27.93,0.0,-4.9,-0.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ndkshidshisbssayacantik""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,-5.19
"""3""","""251230PMP6MTW6""",2025-12-30,2025-12-31,-2.65,14.9,-14.9,0.0,-2.5,-0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gq6pe2qreq""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,-2.65
"""5""","""251230PB5YG8HT""",2025-12-30,2025-12-31,37.32,44.1,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.64,0.0,-2.14,0.0,0.0,"""fiqamokhtar""",4.86,"""SPayLater""","""""","""12x""","""""",0.0,0.0,0.0
"""7""","""251230PB3GD63E""",2025-12-30,2025-12-31,34.51,41.04,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.54,0.0,-1.99,0.0,0.0,"""nurnabilahabukasim""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0
"""10""","""251230PAY9XQWY""",2025-12-30,2025-12-31,26.79,31.5,0.0,4.9,-4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.32,0.0,-1.39,0.0,0.0,"""amyrahbrown""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0,0.0,0.0


In [20]:
balance_all_report = balance_all_normalized.select([
    'Date', 'Order ID'])
#Change the Date time to Date only for easier matching with income report (which only has date, no time)
balance_all_report = balance_all_report.with_columns(
    pl.col("Date").dt.date().alias("Payment Date")
)
balance_all_report = balance_all_report.with_columns(
    pl.col("Payment Date").dt.strftime("%d-%b-%Y").str.to_uppercase().alias("Payment Date")
)
balance_all_report = balance_all_report.with_columns(
    pl.col("Order ID").str.strip_chars(),
    pl.col("Payment Date")
)
balance_all_report.head()

Date,Order ID,Payment Date
datetime[μs],str,str
2026-01-07 23:59:53,"""251227FMQA3XGP""","""07-JAN-2026"""
2026-01-07 23:59:46,"""260102VN9VMVPV""","""07-JAN-2026"""
2026-01-07 23:59:32,"""2601045P6TK6PG""","""07-JAN-2026"""
2026-01-07 23:59:28,"""2601045Y4YTEG6""","""07-JAN-2026"""
2026-01-07 23:59:26,"""2601045E6UGTNU""","""07-JAN-2026"""


In [21]:
report = income_all_normalized.join(
    balance_all_report,
    on="Order ID",
    how="inner",
)
report.head(10)

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026"""
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026"""
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026"""
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026"""
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026"""
"""4176""","""26010575HF5Q6D""",2026-01-05,2026-01-07,14.92,17.52,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.75,0.0,-0.85,0.0,0.0,"""nurainsyazliyana""",4.86,"""SPayLater""","""""","""1x""","""""",0.0,0.0,0.0,2026-01-07 23:59:06,"""07-JAN-2026"""
"""7612""","""26010347C470NX""",2026-01-03,2026-01-07,16.71,19.5,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.05,0.0,-0.74,0.0,0.0,"""farizaalissya""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:02,"""07-JAN-2026"""
"""493""","""260101UAHS8FRR""",2026-01-01,2026-01-07,6.76,7.89,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.83,0.0,-0.3,0.0,0.0,"""shazulkiflii""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:58:33,"""07-JAN-2026"""
"""557""","""2601069VSE7UX0""",2026-01-06,2026-01-07,33.34,38.96,0.0,1.3,-6.3,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.1,0.0,-1.52,0.0,0.0,"""amirahizzati01""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:58:22,"""07-JAN-2026"""


In [22]:
report = report.with_columns(
     pl.col("Date").dt.strftime("%b-%Y").str.to_uppercase().alias("Payment Mth")
)
report.head()


Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026""","""JAN-2026"""
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026""","""JAN-2026"""
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026""","""JAN-2026"""
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026""","""JAN-2026"""
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026""","""JAN-2026"""


In [23]:
sales_report_report = sales_report_normalized.select(pl.col("Order ID", "SalesWorkDate"))
sales_report_report.head()

Order ID,SalesWorkDate
str,date
"""2601044PUM2781""",2026-01-05
"""26010450MFPQE1""",2026-01-05
"""2601045R7NTP28""",2026-01-05
"""260101TUY6HNTV""",2026-01-05
"""260101UF58WU38""",2026-01-05


In [24]:
PaymentDetail= report.join(
    sales_report_report,   
    on="Order ID",
    how="inner"
)
PaymentDetail.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth,SalesWorkDate
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str,date
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026""","""JAN-2026""",2026-01-06
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026""","""JAN-2026""",2026-01-06
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026""","""JAN-2026""",2026-01-06
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026""","""JAN-2026""",2026-01-07
"""4176""","""26010575HF5Q6D""",2026-01-05,2026-01-07,14.92,17.52,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.75,0.0,-0.85,0.0,0.0,"""nurainsyazliyana""",4.86,"""SPayLater""","""""","""1x""","""""",0.0,0.0,0.0,2026-01-07 23:59:06,"""07-JAN-2026""","""JAN-2026""",2026-01-07


In [25]:
income_not_balance = income_all_normalized.join(
    balance_all_normalized,
    on="Order ID",
    how="anti"
)
income_not_balance.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""8832""","""2512222NWK4XVR""",2025-12-22,2025-12-31,0.0,66.0,-66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""aaadrw""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""9076""","""25122108AA8SM7""",2025-12-21,2025-12-31,0.0,39.8,-39.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""rajaindah5597""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""6067""","""251228GK5T85K4""",2025-12-28,2026-01-01,0.0,72.5,-72.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""_azto1n21w""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""7582""","""251225AV1ET5T5""",2025-12-25,2026-01-01,0.0,22.66,-22.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""awimzjr""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0
"""8842""","""2512221YJBXS68""",2025-12-22,2026-01-01,0.0,8.58,-8.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""nuradilanazira""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0


In [26]:
balance_not_income = balance_all_normalized.join(
    income_all_normalized, 
    on="Order ID",
    how="anti"
)
balance_not_income.head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,Payment Amount,Payment Date,Payment Month
datetime[μs],str,str,str,str,str,str,f64,str,f64,str,str
2026-01-07 18:18:01,"""Adjustment""","""Lost Compensation for parcel #…","""251227FHQ8CDPK""","""Money In""","""17.43""","""Transaction Completed""",117084.45,"""Transaction Report""",17.43,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 17:18:27,"""Adjustment""","""Wallet Adjustment due to Buyer…","""251026534124X4""","""Money In""","""4.33""","""Transaction Completed""",99001.06,"""Transaction Report""",4.33,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-9388.4""","""Transaction Completed""",0.0,"""Transaction Report""",-9388.4,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 02:00:10,"""Adjustment""","""Lost Compensation for parcel #…","""251220TKVS7VJY""","""Money In""","""19.28""","""Transaction Completed""",1.3584e6,"""Transaction Report""",19.28,"""07-JAN-2026""","""JAN-2026"""
2026-01-06 19:44:35,"""Adjustment""","""Lost Compensation for parcel #…","""2512100C5MHXUR""","""Money In""","""14.07""","""Transaction Completed""",1.3009e6,"""Transaction Report""",14.07,"""06-JAN-2026""","""JAN-2026"""


## Reconciliation Page

In [27]:
first_part = sales_report_normalized.select(["Order ID", "OrderNum","MarketPlaceOrderNum", "SalesWorkDate","SalesCenterAmount","Sales Month"])
first_part.head()

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month
str,str,str,date,f64,str
"""2601044PUM2781""","""9524949786""","""2601044PUM2781""",2026-01-05,21.0,"""JAN-2026"""
"""26010450MFPQE1""","""9524950020""","""26010450MFPQE1""",2026-01-05,98.29,"""JAN-2026"""
"""2601045R7NTP28""","""9524951072""","""2601045R7NTP28""",2026-01-05,107.65,"""JAN-2026"""
"""260101TUY6HNTV""","""9524928466""","""260101TUY6HNTV""",2026-01-05,9.56,"""JAN-2026"""
"""260101UF58WU38""","""9524930857""","""260101UF58WU38""",2026-01-05,2.73,"""JAN-2026"""


In [28]:
first_part.filter(pl.col("Order ID") == "260101U636TEDR")

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month
str,str,str,date,f64,str
"""260101U636TEDR""","""9524929352""","""260101U636TEDR""",2026-01-05,82.91,"""JAN-2026"""


In [29]:
payment_date = balance_all_normalized.select(["Order ID", "Payment Date", "Payment Month", "Payment Amount"])
payment_date.head()

Order ID,Payment Date,Payment Month,Payment Amount
str,str,str,f64
"""251227FMQA3XGP""","""07-JAN-2026""","""JAN-2026""",13.42
"""260102VN9VMVPV""","""07-JAN-2026""","""JAN-2026""",8.31
"""2601045P6TK6PG""","""07-JAN-2026""","""JAN-2026""",21.25
"""2601045Y4YTEG6""","""07-JAN-2026""","""JAN-2026""",13.19
"""2601045E6UGTNU""","""07-JAN-2026""","""JAN-2026""",23.08


In [30]:
payment_date.filter(pl.col("Order ID") == "2601045SHHRFX7")

Order ID,Payment Date,Payment Month,Payment Amount
str,str,str,f64
"""2601045SHHRFX7""","""14-JAN-2026""","""JAN-2026""",26.56


In [31]:
income_all_normalized.filter(pl.col("Order ID") == "260101U636TEDR")

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""1""","""260101U636TEDR""",2026-01-01,2026-01-06,55.74,87.91,-16.74,5.0,-18.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0,-4.07,0.0,-7.07,0.0,-4.29,0.0,0.0,"""bella_chrisna98""",4.86,"""SPayLater""","""""","""12x""","""WATS0101A""",0.0,-4.07,0.0


In [32]:
thrid_part = income_all_normalized.select(
    "Order ID",
    pl.col("Commission Fee (incl. SST)").alias("Commission Fee"),
    pl.col("Transaction Fee (Incl. SST)").alias("Transaction Fee"),
    pl.col("Service Fee (Incl. SST)").alias("Service Fee"),
    pl.col("AMS Commission Fee"),
    pl.col("Return to Seller Fee").alias("Return QC Fee"),
    pl.col("Voucher Sponsored by Seller").alias("Voucher/(Disc rebate)"),
    pl.col("Refund Amount").alias("Refund"),
    (pl.col('Shipping Fee Paid by Buyer (excl. SST)') + pl.col('Shipping Fee Charged by Logistic Provider')+pl.col('Seller Paid Shipping Fee SST') + pl.col('Shipping Rebate From Shopee') + pl.col('Reverse Shipping Fee') + pl.col('Reverse Shipping Fee SST')).alias("Actual Shipping Fee")
)

In [33]:
thrid_part.filter(pl.col("Order ID") == "260101U636TEDR")

Order ID,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee
str,f64,f64,f64,f64,f64,f64,f64,f64
"""260101U636TEDR""",-7.07,-4.29,0.0,0.0,0.0,-4.07,-16.74,0.0


In [34]:
recon_report = first_part.join(payment_date, on=["Order ID"], how="inner") \
                         .join(thrid_part, on=["Order ID"], how="inner")
recon_report.columns

['Order ID',
 'OrderNum',
 'MarketPlaceOrderNum',
 'SalesWorkDate',
 'SalesCenterAmount',
 'Sales Month',
 'Payment Date',
 'Payment Month',
 'Payment Amount',
 'Commission Fee',
 'Transaction Fee',
 'Service Fee',
 'AMS Commission Fee',
 'Return QC Fee',
 'Voucher/(Disc rebate)',
 'Refund',
 'Actual Shipping Fee']

In [35]:
recon_report = recon_report.with_columns(
    (pl.col("SalesCenterAmount") - pl.col("Payment Amount") + pl.col("Commission Fee") 
     + pl.col("Transaction Fee") + pl.col("Service Fee") + pl.col("AMS Commission Fee")
     + pl.col("Return QC Fee") + pl.col("Voucher/(Disc rebate)") + pl.col("Actual Shipping Fee")).alias("Outstanding")
)
recon_report.head()

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month,Payment Date,Payment Month,Payment Amount,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""260104709M2A43""","""9524955631""","""260104709M2A43""",2026-01-05,14.45,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",12.3,-1.6,-0.55,0.0,0.0,0.0,0.0,0.0,0.0,-1.5543e-15
"""2601046SQV3UDM""","""9524954642""","""2601046SQV3UDM""",2026-01-05,25.6,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",21.93,-2.7,-0.97,0.0,0.0,0.0,0.0,0.0,0.0,1.5543e-15
"""2601046QDJN3M6""","""9524954331""","""2601046QDJN3M6""",2026-01-05,48.96,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",41.95,-5.16,-1.85,0.0,0.0,0.0,0.0,0.0,0.0,-2.2204e-15
"""2601046DKNKUTR""","""9524953197""","""2601046DKNKUTR""",2026-01-05,26.45,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",-5.19,0.0,0.0,0.0,0.0,0.0,0.0,-26.45,-5.19,26.45
"""2601045TD8KG91""","""9524951263""","""2601045TD8KG91""",2026-01-05,61.51,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",52.04,-6.48,-2.99,0.0,0.0,0.0,0.0,0.0,0.0,-1.7764e-15


In [36]:
report_not_recon = report.join(
    recon_report.select("Order ID").unique(),
    on="Order ID",
    how="anti"
)

print("entries in report but not in recon:", report_not_recon.height)
report_not_recon.head()

entries in report but not in recon: 45018


Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026""","""JAN-2026"""
"""493""","""260101UAHS8FRR""",2026-01-01,2026-01-07,6.76,7.89,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.83,0.0,-0.3,0.0,0.0,"""shazulkiflii""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:58:33,"""07-JAN-2026""","""JAN-2026"""
"""2377""","""2512235NA29GPW""",2025-12-23,2026-01-07,90.2,114.14,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,-9.0,0.0,-10.97,0.0,-3.97,0.0,0.0,"""aneezarr""",3.78,"""Credit / Debit Card""","""""","""""","""WATS1223A""",0.0,-9.0,0.0,2026-01-07 23:58:05,"""07-JAN-2026""","""JAN-2026"""
"""8792""","""260102VY53NBCR""",2026-01-02,2026-01-07,16.97,19.8,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.08,0.0,-0.75,0.0,0.0,"""aynafateen""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:57:22,"""07-JAN-2026""","""JAN-2026"""
"""2119""","""251225AHWVQSX0""",2025-12-25,2026-01-07,13.82,16.13,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.7,0.0,-0.61,0.0,0.0,"""69wu9rfuyf""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:46:10,"""07-JAN-2026""","""JAN-2026"""


In [37]:
recon_report.columns

['Order ID',
 'OrderNum',
 'MarketPlaceOrderNum',
 'SalesWorkDate',
 'SalesCenterAmount',
 'Sales Month',
 'Payment Date',
 'Payment Month',
 'Payment Amount',
 'Commission Fee',
 'Transaction Fee',
 'Service Fee',
 'AMS Commission Fee',
 'Return QC Fee',
 'Voucher/(Disc rebate)',
 'Refund',
 'Actual Shipping Fee',
 'Outstanding']

## Comparison with original data

### Compare the generated recon report with the original recon reprot 

In [38]:
path = './Shopee Sample Reports (Testing)/Shopee Payment Master List 180126.xlsx'
ori_recon = pl.read_excel(path, sheet_name="Recon", has_header=True)
ori_recon.head()

Could not determine dtype for column 19, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 25, falling back to string
Could not determine dtype for column 26, falling back to string
Could not determine dtype for column 27, falling back to string
Could not determine dtype for column 28, falling back to string
Could not determine dtype for column 29, falling back to string
Could not determine dtype for column 30, falling back to string
Could not determine dtype for column 31, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 33, falling back to string
Could not determine dtype for column 34, falling back to string
Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 38,

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str
"""9523929588""","""250705B8MM4UH4""","""2025-07-06""",37.18,2025-07-01,2025-07-23,2025-07-01,23.45,2.87,1.61,0.0,0.0,0,0.0,0.0,0.0,9.25,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9.25,null
"""9523802829""","""250612C1TJ9SHE""","""2025-07-07""",-3.0,2025-07-01,2025-06-25,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-3.0,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-3.0,null
"""9523757815""","""250604N6AD379X""","""2025-07-07""",-9.73,2025-07-01,2025-06-18,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-9.73,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-9.73,null
"""9523971310""","""250711TY92VWPU""","""2025-07-13""",153.78,2025-07-01,2025-07-23,2025-07-01,128.05,15.48,7.36,0.0,0.0,0,11.0,0.0,8.8818e-16,2.89,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.89,null
"""9523975421""","""2507120F7GFHVV""","""2025-07-14""",278.03,2025-07-01,2025-07-30,2025-08-01,237.96,28.53,10.59,0.0,0.0,0,16.0,0.0,0.0,0.95,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.95,null


In [39]:
ori_recon = ori_recon.select(pl.col("MarketPlaceOrderNum", "OrderNum", "WorkDate", "SalesCenterAmount", "Pymt Amt", "Pymt Date", "Commission Fee", "Transaction Fee", "Service Fee", "AMS Commission Fee", "Return QC Fee", "Voucher/(Disc rebate)", "Refund", "Actual Shipping Fee", "Outstanding"))

ori_renamed = ori_recon.with_columns(
     pl.col("MarketPlaceOrderNum").str.strip_chars().alias("Order ID Ori"),
     pl.col("OrderNum").str.strip_chars().alias("OrderNum Ori"),
     pl.col("MarketPlaceOrderNum").str.strip_chars().alias("MarketPlaceOrderNum Ori"),
     pl.col("WorkDate").str.strptime(pl.Date, '%Y-%m-%d', strict=False).alias("SalesWorkDate Ori"),
     pl.col("SalesCenterAmount").cast(pl.Float64, strict=False).alias("SalesCenterAmount Ori"),
     pl.col("Pymt Amt").cast(pl.Float64, strict=False).alias("Payment Amount Ori"),
     pl.col("Pymt Date").dt.strftime("%b-%Y").str.to_uppercase().alias("Payment Month Ori"),
     pl.col("Commission Fee").cast(pl.Float64, strict=False).alias("Commission Fee Ori"),
     pl.col("Transaction Fee").cast(pl.Float64, strict=False).alias("Transaction Fee Ori"),
     pl.col("Service Fee").cast(pl.Float64, strict=False).alias("Service Fee Ori"),
     pl.col("AMS Commission Fee").cast(pl.Float64, strict=False).alias("AMS Commission Fee Ori"),
     pl.col("Return QC Fee").cast(pl.Float64, strict=False).alias("Return QC Fee Ori"),
     pl.col("Voucher/(Disc rebate)").cast(pl.Float64, strict=False).alias("Voucher/(Disc rebate) Ori"), 
     pl.col("Refund").cast(pl.Float64, strict=False).alias("Refund Ori"),
     pl.col("Actual Shipping Fee").cast(pl.Float64, strict=False).alias("Actual Shipping Fee Ori"),
     pl.col("Outstanding").cast(pl.Float64, strict=False).alias("Outstanding Ori"), 
     )
ori_renamed = ori_renamed.drop([
          "MarketPlaceOrderNum",
          "OrderNum",
          "WorkDate",
          "SalesCenterAmount",
          "Pymt Amt",
          "Pymt Date",
          "Commission Fee",
          "Transaction Fee",
          "Service Fee",
          "AMS Commission Fee",
          "Return QC Fee",
          "Voucher/(Disc rebate)",
          "Refund",
          "Actual Shipping Fee",
          "Outstanding",
     ])

In [40]:
ori_renamed.columns

['Order ID Ori',
 'OrderNum Ori',
 'MarketPlaceOrderNum Ori',
 'SalesWorkDate Ori',
 'SalesCenterAmount Ori',
 'Payment Amount Ori',
 'Payment Month Ori',
 'Commission Fee Ori',
 'Transaction Fee Ori',
 'Service Fee Ori',
 'AMS Commission Fee Ori',
 'Return QC Fee Ori',
 'Voucher/(Disc rebate) Ori',
 'Refund Ori',
 'Actual Shipping Fee Ori',
 'Outstanding Ori']

In [41]:
cols = [
"Order ID",
"OrderNum",
"MarketPlaceOrderNum",
"SalesWorkDate",
"SalesCenterAmount",
"Payment Amount",
"Payment Month",
"Commission Fee",
"Transaction Fee",
"Service Fee",
"AMS Commission Fee",
"Return QC Fee",
"Voucher/(Disc rebate)",
"Refund",
"Actual Shipping Fee",
"Outstanding"
]

# rename columns
recon_renamed = recon_report.rename({c: f"{c} Recon" for c in cols})
#ori_renamed = ori_recon.rename({c: f"{c} Ori" for c in cols})

# join using Order ID
compare = recon_renamed.join(
    ori_renamed,
    left_on="Order ID Recon",
    right_on="Order ID Ori",
    how="inner"
)

In [42]:
compare.head()

Order ID Recon,OrderNum Recon,MarketPlaceOrderNum Recon,SalesWorkDate Recon,SalesCenterAmount Recon,Sales Month,Payment Date,Payment Month Recon,Payment Amount Recon,Commission Fee Recon,Transaction Fee Recon,Service Fee Recon,AMS Commission Fee Recon,Return QC Fee Recon,Voucher/(Disc rebate) Recon,Refund Recon,Actual Shipping Fee Recon,Outstanding Recon,OrderNum Ori,MarketPlaceOrderNum Ori,SalesWorkDate Ori,SalesCenterAmount Ori,Payment Amount Ori,Payment Month Ori,Commission Fee Ori,Transaction Fee Ori,Service Fee Ori,AMS Commission Fee Ori,Return QC Fee Ori,Voucher/(Disc rebate) Ori,Refund Ori,Actual Shipping Fee Ori,Outstanding Ori
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,date,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2601033WTQVRED""","""9524947374""","""2601033WTQVRED""",2026-01-05,91.0,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",78.27,-9.09,-3.64,0.0,0.0,0.0,-9.0,0.0,0.0,-9.0,"""9524947374""","""2601033WTQVRED""",null,91.0,78.27,"""JAN-2026""",9.09,3.64,0.0,0.0,0.0,9.0,0.0,0.0,3.9968e-15
"""26010339RXJMXH""","""9524945618""","""26010339RXJMXH""",2026-01-05,18.4,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",15.57,-1.94,-0.89,0.0,0.0,0.0,0.0,0.0,0.0,-1.6653e-15,"""9524945618""","""26010339RXJMXH""",null,18.4,15.57,"""JAN-2026""",1.94,0.89,0.0,0.0,0.0,0.0,0.0,0.0,-1.6653e-15
"""2601033CEW09QH""","""9524945867""","""2601033CEW09QH""",2026-01-05,41.8,"""JAN-2026""","""06-JAN-2026""","""JAN-2026""",35.22,-4.55,-2.03,0.0,0.0,0.0,0.0,0.0,0.0,-1.3323e-15,"""9524945867""","""2601033CEW09QH""",null,41.8,35.22,"""JAN-2026""",4.55,2.03,0.0,0.0,0.0,0.0,0.0,0.0,-1.3323e-15
"""2601045KBCQ29G""","""9524950663""","""2601045KBCQ29G""",2026-01-05,29.25,"""JAN-2026""","""08-JAN-2026""","""JAN-2026""",25.06,-3.08,-1.11,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15,"""9524950663""","""2601045KBCQ29G""",null,29.25,25.06,"""JAN-2026""",3.08,1.11,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15
"""2601020URJ2N7E""","""9524939608""","""2601020URJ2N7E""",2026-01-05,12.89,"""JAN-2026""","""06-JAN-2026""","""JAN-2026""",11.11,-1.29,-0.49,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15,"""9524939608""","""2601020URJ2N7E""",null,12.89,11.11,"""JAN-2026""",1.29,0.49,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15


In [43]:
cols = [
"OrderNum",
"MarketPlaceOrderNum",
"SalesWorkDate",
"SalesCenterAmount",
"Payment Amount",
"Payment Month",
"Commission Fee",
"Transaction Fee",
"Service Fee",
"AMS Commission Fee",
"Return QC Fee",
"Voucher/(Disc rebate)",
"Refund",
"Actual Shipping Fee",
"Outstanding"
]
ordered_cols = []
for c in cols:
    ordered_cols.append(f"{c} Recon")
    ordered_cols.append(f"{c} Ori")

compare = compare.select(ordered_cols)

In [44]:
compare.head()

OrderNum Recon,OrderNum Ori,MarketPlaceOrderNum Recon,MarketPlaceOrderNum Ori,SalesWorkDate Recon,SalesWorkDate Ori,SalesCenterAmount Recon,SalesCenterAmount Ori,Payment Amount Recon,Payment Amount Ori,Payment Month Recon,Payment Month Ori,Commission Fee Recon,Commission Fee Ori,Transaction Fee Recon,Transaction Fee Ori,Service Fee Recon,Service Fee Ori,AMS Commission Fee Recon,AMS Commission Fee Ori,Return QC Fee Recon,Return QC Fee Ori,Voucher/(Disc rebate) Recon,Voucher/(Disc rebate) Ori,Refund Recon,Refund Ori,Actual Shipping Fee Recon,Actual Shipping Fee Ori,Outstanding Recon,Outstanding Ori
str,str,str,str,date,date,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""9524947374""","""9524947374""","""2601033WTQVRED""","""2601033WTQVRED""",2026-01-05,null,91.0,91.0,78.27,78.27,"""JAN-2026""","""JAN-2026""",-9.09,9.09,-3.64,3.64,0.0,0.0,0.0,0.0,0.0,0.0,-9.0,9.0,0.0,0.0,0.0,0.0,-9.0,3.9968e-15
"""9524945618""","""9524945618""","""26010339RXJMXH""","""26010339RXJMXH""",2026-01-05,null,18.4,18.4,15.57,15.57,"""JAN-2026""","""JAN-2026""",-1.94,1.94,-0.89,0.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.6653e-15,-1.6653e-15
"""9524945867""","""9524945867""","""2601033CEW09QH""","""2601033CEW09QH""",2026-01-05,null,41.8,41.8,35.22,35.22,"""JAN-2026""","""JAN-2026""",-4.55,4.55,-2.03,2.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.3323e-15,-1.3323e-15
"""9524950663""","""9524950663""","""2601045KBCQ29G""","""2601045KBCQ29G""",2026-01-05,null,29.25,29.25,25.06,25.06,"""JAN-2026""","""JAN-2026""",-3.08,3.08,-1.11,1.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15,1.1102e-15
"""9524939608""","""9524939608""","""2601020URJ2N7E""","""2601020URJ2N7E""",2026-01-05,null,12.89,12.89,11.11,11.11,"""JAN-2026""","""JAN-2026""",-1.29,1.29,-0.49,0.49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.1102e-15,1.1102e-15


In [45]:
recon_report.head()

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month,Payment Date,Payment Month,Payment Amount,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""260104709M2A43""","""9524955631""","""260104709M2A43""",2026-01-05,14.45,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",12.3,-1.6,-0.55,0.0,0.0,0.0,0.0,0.0,0.0,-1.5543e-15
"""2601046SQV3UDM""","""9524954642""","""2601046SQV3UDM""",2026-01-05,25.6,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",21.93,-2.7,-0.97,0.0,0.0,0.0,0.0,0.0,0.0,1.5543e-15
"""2601046QDJN3M6""","""9524954331""","""2601046QDJN3M6""",2026-01-05,48.96,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",41.95,-5.16,-1.85,0.0,0.0,0.0,0.0,0.0,0.0,-2.2204e-15
"""2601046DKNKUTR""","""9524953197""","""2601046DKNKUTR""",2026-01-05,26.45,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",-5.19,0.0,0.0,0.0,0.0,0.0,0.0,-26.45,-5.19,26.45
"""2601045TD8KG91""","""9524951263""","""2601045TD8KG91""",2026-01-05,61.51,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",52.04,-6.48,-2.99,0.0,0.0,0.0,0.0,0.0,0.0,-1.7764e-15


## Export the Result Out to Excel File

In [46]:
import xlsxwriter, datetime

output_path = "output_formatted_v2.xlsx"
workbook = xlsxwriter.Workbook(output_path)

# ═══════════════════════════════════════════════════════════════
# FORMAT DEFINITIONS
# ═══════════════════════════════════════════════════════════════
def _hdr(bg, fc='white', sz=11):
    return workbook.add_format({
        'bold': True, 'font_color': fc, 'bg_color': bg,
        'border': 1, 'align': 'center', 'valign': 'vcenter',
        'font_size': sz, 'text_wrap': True,
    })

# Level-1 (top banner) per group name
L1_FMT = {
    'Order Info':              _hdr('#4472C4'),
    'Released Amount Details': _hdr('#C65911'),
    'Vouchers and Rebates':    _hdr('#B4A7D6', fc='black'),
    'Fees and Charges':        _hdr('#A9D18E', fc='black'),
    'Buyer Info':              _hdr('#7030A0'),
    'Reference Info':          _hdr('#C65911'),
}
L1_DEFAULT = _hdr('#C65911')
L1_BLANK   = workbook.add_format({'border': 1, 'bg_color': '#FFFFFF'})

# Level-2 (sub-group banner) – light orange, dark text
L2_FMT   = _hdr('#F4B084', fc='black', sz=10)
L2_BLANK = workbook.add_format({'border': 1, 'bg_color': '#FFFFFF'})

# Level-3 column-name row – standard orange
fmt_col = _hdr('#ED7D31', sz=10)

# Recon / generic header formats
fmt_group  = _hdr('#C65911')
fmt_header = _hdr('#ED7D31', sz=10)

# Data cell formats
fmt_text = workbook.add_format({'border': 1, 'font_size': 10, 'valign': 'vcenter'})
fmt_num  = workbook.add_format({
    'border': 1, 'font_size': 10, 'valign': 'vcenter',
    'num_format': '#,##0.00;(#,##0.00);"-"',
})
fmt_date = workbook.add_format({
    'border': 1, 'font_size': 10, 'valign': 'vcenter', 'num_format': 'DD-MMM-YY',
})
fmt_month = workbook.add_format({
    'border': 1, 'font_size': 10, 'valign': 'vcenter',
    'num_format': 'MMM-YY', 'align': 'center',
})
fmt_text_center = workbook.add_format({
    'border': 1, 'font_size': 10, 'valign': 'vcenter', 'align': 'center',
})


# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════
MONEY_COLS = {
    'SalesCenterAmount', 'Payment Amount', 'Total Released Amount (RM)',
    'Commission Fee', 'Transaction Fee', 'Service Fee',
    'Return QC Fee', 'Voucher/(Disc rebate)', 'Refund',
    'Actual Shipping Fee', 'Outstanding',
    'Product Price', 'Refund Amount', 'Net Voucher', 'Net Shipping Fees',
    'Shipping Fee Paid by Buyer (excl. SST)', 'Shipping Fee Charged by Logistic Provider',
    'Seller Paid Shipping Fee SST', 'Shipping Rebate From Shopee',
    'Reverse Shipping Fee', 'Reverse Shipping Fee SST',
    'Saver Programme Shipping Fee Savings', 'Return to Seller Fee',
    'Rebate Provided by Shopee', 'Voucher Sponsored by Seller',
    'Coin Cashback Sponsored by Seller', 'Commission Fee (incl. SST)',
    'Service Fee (Incl. SST)', 'Transaction Fee (Incl. SST)',
    'AMS Commission Fee', 'Saver Programme Fee (Incl. SST)',
    'Transaction Fee Rate (%)', 'Lost Compensation', 'Amount',
    'Balance After Transactions',
}
DATE_COLS  = {'SalesWorkDate', 'WorkDate', 'Order Creation Date', 'Payout Completed Date', 'Date'}
MONTH_COLS = {'Sales Month', 'Payment Month', 'Payment Mth'}

def col_format(c):
    if c in MONEY_COLS:  return fmt_num
    if c in DATE_COLS:   return fmt_date
    if c in MONTH_COLS:  return fmt_month
    if c == 'Payment Date': return fmt_text_center
    return fmt_text

def write_value(ws, r, c, v, fmt):
    if v is None:
        ws.write_blank(r, c, None, fmt)
    elif isinstance(v, (datetime.date, datetime.datetime)):
        ws.write_datetime(r, c, v, fmt)
    elif isinstance(v, (int, float)):
        ws.write_number(r, c, v, fmt)
    else:
        ws.write_string(r, c, str(v), fmt)


# ═══════════════════════════════════════════════════════════════
# REPORT SHEET — 3-level grouped headers
# ═══════════════════════════════════════════════════════════════
def write_report_sheet(wb, sheet_name, df):
    ws = wb.add_worksheet(sheet_name)

    # (level1, level2, [columns])  — order defines column sequence
    LAYOUT = [
        ('Order Info', '', [
            'Sequence No.', 'Order ID', 'Order Creation Date', 'Payout Completed Date']),
        ('Released Amount Details', 'Order Income', [
            'Total Released Amount (RM)']),
        ('Released Amount Details', 'Merchandise', [
            'Product Price', 'Refund Amount']),
        ('Released Amount Details', 'Shipping', [
            'Shipping Fee Paid by Buyer (excl. SST)',
            'Shipping Fee Charged by Logistic Provider',
            'Seller Paid Shipping Fee SST',
            'Shipping Rebate From Shopee',
            'Reverse Shipping Fee',
            'Reverse Shipping Fee SST',
            'Saver Programme Shipping Fee Savings',
            'Return to Seller Fee']),
        ('Vouchers and Rebates', '', [
            'Rebate Provided by Shopee',
            'Voucher Sponsored by Seller',
            'Coin Cashback Sponsored by Seller']),
        ('Fees and Charges', '', [
            'Commission Fee (incl. SST)',
            'Service Fee (Incl. SST)',
            'Transaction Fee (Incl. SST)',
            'AMS Commission Fee',
            'Saver Programme Fee (Incl. SST)']),
        ('Buyer Info', '', [
            'Username (Buyer)',
            'Transaction Fee Rate (%)',
            'Buyer Payment Method',
            'Buyer Payment Method Details_1(if applicable)',
            'Payment Details / Installment Plan']),
        ('Reference Info', 'Promotion', [
            'Voucher Code From Seller']),
        ('Reference Info', 'Compensation', [
            'Lost Compensation']),
        ('', '', [
            'Date', 'Payment Date', 'Payment Mth',
            'Net Voucher', 'Net Shipping Fees']),
    ]

    # Flatten & keep only columns that exist in df
    flat = []  # list of (l1, l2, col_name)
    for l1, l2, cols in LAYOUT:
        for c in cols:
            if c in df.columns:
                flat.append((l1, l2, c))
    # Append any remaining columns not covered by LAYOUT
    used = {f[2] for f in flat}
    skip = {'_source_sheet', '_source_file'}
    for c in df.columns:
        if c not in used and c not in skip:
            flat.append(('', '', c))

    col_order = [f[2] for f in flat]
    N = len(col_order)

    # ── Row 0: Level-1 merged banners ──
    i = 0
    while i < N:
        l1 = flat[i][0]
        j = i + 1
        while j < N and flat[j][0] == l1:
            j += 1
        if l1:
            fmt = L1_FMT.get(l1, L1_DEFAULT)
            if j - i == 1:
                ws.write(0, i, l1, fmt)
            else:
                ws.merge_range(0, i, 0, j - 1, l1, fmt)
        else:
            for k in range(i, j):
                ws.write_blank(0, k, None, L1_BLANK)
        i = j

    # ── Row 1: Level-2 merged banners ──
    i = 0
    while i < N:
        l1, l2 = flat[i][0], flat[i][1]
        j = i + 1
        while j < N and flat[j][0] == l1 and flat[j][1] == l2:
            j += 1
        if l2:
            if j - i == 1:
                ws.write(1, i, l2, L2_FMT)
            else:
                ws.merge_range(1, i, 1, j - 1, l2, L2_FMT)
        else:
            for k in range(i, j):
                ws.write_blank(1, k, None, L2_BLANK)
        i = j

    # ── Row 2: Column names ──
    for i, cn in enumerate(col_order):
        ws.write(2, i, cn, fmt_col)

    # ── Row 3+: Data ──
    for ri, row in enumerate(df.select(col_order).iter_rows()):
        for ci, v in enumerate(row):
            write_value(ws, ri + 3, ci, v, col_format(col_order[ci]))

    # Autofilter on column-name row
    ws.autofilter(2, 0, df.height + 2, N - 1)
    for i, cn in enumerate(col_order):
        ws.set_column(i, i, max(len(cn) + 2, 14))
    ws.freeze_panes(3, 0)
    ws.set_row(0, 22)
    ws.set_row(1, 22)
    ws.set_row(2, 40)


# ═══════════════════════════════════════════════════════════════
# RECONCILIATION SHEET — 2-level grouped headers
# ═══════════════════════════════════════════════════════════════
def write_recon_sheet(wb, sheet_name, df):
    ws = wb.add_worksheet(sheet_name)
    groups = [
        ('Order Info',        ['Order ID', 'OrderNum', 'MarketPlaceOrderNum']),
        ('Sales Center',      ['SalesWorkDate', 'SalesCenterAmount', 'Sales Month']),
        ('Payment Info',      ['Payment Date', 'Payment Month', 'Payment Amount']),
        ('Fees & Deductions', ['Commission Fee', 'Transaction Fee', 'Service Fee',
                               'Return QC Fee', 'Voucher/(Disc rebate)', 'Refund',
                               'Actual Shipping Fee']),
        ('Result',            ['Outstanding']),
    ]
    col_order = []
    for _, cols in groups:
        for c in cols:
            if c in df.columns:
                col_order.append(c)
    N = len(col_order)

    idx = 0
    for gl, cols in groups:
        present = [c for c in cols if c in df.columns]
        if not present: continue
        if len(present) == 1:
            ws.write(0, idx, gl, fmt_group)
        else:
            ws.merge_range(0, idx, 0, idx + len(present) - 1, gl, fmt_group)
        idx += len(present)

    for i, cn in enumerate(col_order):
        ws.write(1, i, cn, fmt_header)
    for ri, row in enumerate(df.select(col_order).iter_rows()):
        for ci, v in enumerate(row):
            write_value(ws, ri + 2, ci, v, col_format(col_order[ci]))

    ws.autofilter(1, 0, df.height + 1, N - 1)
    for i, cn in enumerate(col_order):
        ws.set_column(i, i, max(len(cn) + 2, 14))
    ws.freeze_panes(2, 0)
    ws.set_row(0, 25)
    ws.set_row(1, 35)


# ═══════════════════════════════════════════════════════════════
# GENERIC STYLED SHEET (single header row)
# ═══════════════════════════════════════════════════════════════
def write_styled_sheet(wb, sheet_name, df):
    ws = wb.add_worksheet(sheet_name)
    cols = [c for c in df.columns if not c.startswith('_source')]
    for i, cn in enumerate(cols):
        ws.write(0, i, cn, fmt_header)
    for ri, row in enumerate(df.select(cols).iter_rows()):
        for ci, v in enumerate(row):
            write_value(ws, ri + 1, ci, v, col_format(cols[ci]))
    ws.autofilter(0, 0, df.height, len(cols) - 1)
    for i, cn in enumerate(cols):
        ws.set_column(i, i, max(len(cn) + 2, 14))
    ws.freeze_panes(1, 0)
    ws.set_row(0, 35)


# ═══════════════════════════════════════════════════════════════
# WRITE ALL SHEETS
# ═══════════════════════════════════════════════════════════════
write_report_sheet(workbook,  'Report',               report)
write_recon_sheet(workbook,   'Reconciliation',       recon_report)
write_styled_sheet(workbook,  'Income not in Balance', income_not_balance)
write_styled_sheet(workbook,  'Balance not in Income', balance_not_income)

workbook.close()
print(f'Exported to {output_path}')

Exported to output_formatted_v2.xlsx
